#### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

#### Loading the Dataset

In [3]:
data_path = Path("../data/raw/Online Retail.xlsx")
df = pd.read_excel(data_path, engine="openpyxl")
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


#### Inspecting Dataset Dimensions

In [4]:
print(f"Number of Rows: {df.shape[0]:,}")
print(f"Number of Columns: {df.shape[1]}")
print(f"Total Data Points: {df.size:,}")

Number of Rows: 541,909
Number of Columns: 8
Total Data Points: 4,335,272


#### Inspecting Dataset Information

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


- Temporal information: InvoiceDate is already stored as a datetime variable, which is essential for chronological analysis and daily revenue aggregation.

- Missing values: Description contains 1,454 missing values, while CustomerID contains 135,080. We'll investigate their relevance before deciding how to handle them.

- Revenue calculation: Quantity and UnitPrice provide the necessary information to calculate transaction-level revenue. However, we'll need to examine cancellations, returns, and invalid transactions before defining our forecasting target.

#### Inspecting Column Names and Data Types

In [7]:
column_summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (df.isna().mean() * 100).round(2).values
})
display(column_summary)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage
0,InvoiceNo,object,541909,0,0.00
1,StockCode,object,541909,0,0.00
2,Description,object,540455,1454,0.27
3,Quantity,int64,541909,0,0.00
4,InvoiceDate,datetime64[us],541909,0,0.00
5,UnitPrice,float64,541909,0,0.00
6,CustomerID,float64,406829,135080,24.93
7,Country,str,541909,0,0.00


#### Inspecting Numerical Feature Distributions

In [8]:
display(df[["Quantity", "UnitPrice"]].describe().T)

,count,mean,std,min,25%,50%,75%,max
Quantity,"541,909.00",9.55,218.08,"-80,995.00",1.00,3.00,10.00,"80,995.00"
UnitPrice,"541,909.00",4.61,96.76,"-11,062.06",1.25,2.08,4.13,"38,970.00"


**Key Observations**
- **Transaction quantities exhibit substantial variability**, with a median of 3 units and extreme values ranging from -80,995 to 80,995, suggesting possible returns, cancellations, or unusually large transactions.
- **Unit prices show considerable dispersion**, with a median of 2.08 and values ranging from -11,062.06 to 38,970.00, indicating potential pricing adjustments or exceptional transactions that require further investigation.

#### Inspecting Negative, Zero, and Positive Transaction Values

In [9]:
for column in ["Quantity", "UnitPrice"]:
    print(f"\n{column} Distribution:")
    summary = pd.DataFrame({
        "Count": [
            (df[column] < 0).sum(),
            (df[column] == 0).sum(),
            (df[column] > 0).sum()
        ]
    }, index=["Negative", "Zero", "Positive"])
    summary["Percentage"] = (summary["Count"] / len(df) * 100).round(2)
    display(summary)


Quantity Distribution:


,Count,Percentage
Negative,10624,1.96
Zero,0,0.00
Positive,531285,98.04



UnitPrice Distribution:


,Count,Percentage
Negative,2,0.00
Zero,2515,0.46
Positive,539392,99.54


**Key Observations**
- **Negative quantities account for 1.96% of transaction records (10,624 entries)**, potentially representing product returns or cancellations, while 98.04% of records contain positive quantities.
- **Unit prices are predominantly positive (99.54%)**, although 2,515 records contain zero prices and 2 records contain negative prices, requiring further investigation before revenue aggregation.

#### Investigating Invoice Cancellation Patterns

In [10]:
invoice_cancellations = df["InvoiceNo"].astype(str).str.startswith("C")
cancellation_summary = pd.DataFrame({
    "Count": [
        invoice_cancellations.sum(),
        (~invoice_cancellations).sum()
    ]
}, index=["Cancelled Invoice Records", "Other Invoice Records"])
cancellation_summary["Percentage"] = (
    cancellation_summary["Count"] / len(df) * 100
).round(2)
display(cancellation_summary)
print(f"Unique Cancelled Invoices: {df.loc[invoice_cancellations, 'InvoiceNo'].nunique():,}")
print(f"Negative Quantities in Cancelled Invoices: {((df['Quantity'] < 0) & invoice_cancellations).sum():,}")
print(f"Negative Quantities in Other Invoices: {((df['Quantity'] < 0) & ~invoice_cancellations).sum():,}")

,Count,Percentage
Cancelled Invoice Records,9288,1.71
Other Invoice Records,532621,98.29


Unique Cancelled Invoices: 3,836
Negative Quantities in Cancelled Invoices: 9,288
Negative Quantities in Other Invoices: 1,336


**Key Observations**
- **Cancelled invoices account for 9,288 transaction records (1.71%) across 3,836 unique invoices**, confirming that cancellations represent a relatively small but meaningful portion of the dataset.
- **All 9,288 cancellation records contain negative quantities**, while an additional 1,336 negative-quantity records occur outside explicitly cancelled invoices, indicating that negative quantities cannot be identified solely through invoice cancellation prefixes.

#### Investigating Negative Quantities Outside Cancelled Invoices

In [11]:
non_cancelled_negative = df.loc[
    (df["Quantity"] < 0) & (~invoice_cancellations)
].copy()
print(f"Total Records: {len(non_cancelled_negative):,}")
print(f"Unique Invoices: {non_cancelled_negative['InvoiceNo'].nunique():,}")
print(f"Missing Descriptions: {non_cancelled_negative['Description'].isna().sum():,}")
print(f"Missing Customer IDs: {non_cancelled_negative['CustomerID'].isna().sum():,}")
print("\nMost Frequent Stock Codes:")
display(non_cancelled_negative["StockCode"].value_counts().head(10))
print("\nSample Records:")
display(non_cancelled_negative.head(10))

Total Records: 1,336
Unique Invoices: 1,336
Missing Descriptions: 862
Missing Customer IDs: 1,336

Most Frequent Stock Codes:


StockCode
85175     5
21830     5
85172     4
22719     4
82494L    4
72802C    4
21161     3
22162     3
22034     3
20966     3
Name: count, dtype: int64


Sample Records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.00,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.00,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.00,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.00,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.00,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.00,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.00,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.00,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.00,NaN,United Kingdom


**Key Observations**
- **All 1,336 negative-quantity records outside cancelled invoices have missing Customer IDs**, while 862 records also lack product descriptions, suggesting that these entries may represent internal adjustments rather than conventional customer transactions.
- **Each record belongs to a unique invoice**, and the sample transactions contain zero unit prices, highlighting the need to distinguish non-standard inventory movements from revenue-generating sales.

#### Investigating Unit Prices of Non-Cancelled Negative Transactions

In [12]:
price_summary = pd.DataFrame({
    "Count": [
        (non_cancelled_negative["UnitPrice"] < 0).sum(),
        (non_cancelled_negative["UnitPrice"] == 0).sum(),
        (non_cancelled_negative["UnitPrice"] > 0).sum()
    ]
}, index=["Negative", "Zero", "Positive"])
price_summary["Percentage"] = (
    price_summary["Count"] / len(non_cancelled_negative) * 100
).round(2)
display(price_summary)
print("\nUnit Price Statistics:")
display(non_cancelled_negative["UnitPrice"].describe())
print(f"\nTotal Monetary Impact: {(non_cancelled_negative['Quantity'] * non_cancelled_negative['UnitPrice']).sum():,.2f}")

,Count,Percentage
Negative,0,0.00
Zero,1336,100.00
Positive,0,0.00



Unit Price Statistics:


count   1,336.00
mean        0.00
std         0.00
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         0.00
Name: UnitPrice, dtype: float64


Total Monetary Impact: 0.00


**Key Observations**
- **All 1,336 negative-quantity records outside cancelled invoices have zero unit prices**, confirming that these transactions contribute no monetary value when revenue is calculated as Quantity × UnitPrice.
- **These records appear consistent with non-revenue inventory adjustments rather than conventional sales**, supporting their exclusion from revenue-generating transactions during data preparation, subject to final validation.

#### Investigating Zero and Negative Unit Prices

In [13]:
non_positive_prices = df.loc[df["UnitPrice"] <= 0].copy()
print(f"Total Records: {len(non_positive_prices):,}")
print(f"Zero-Priced Records: {(non_positive_prices['UnitPrice'] == 0).sum():,}")
print(f"Negative-Priced Records: {(non_positive_prices['UnitPrice'] < 0).sum():,}")
print(f"Missing Descriptions: {non_positive_prices['Description'].isna().sum():,}")
print(f"Missing Customer IDs: {non_positive_prices['CustomerID'].isna().sum():,}")
print("\nQuantity Distribution:")
display(non_positive_prices["Quantity"].describe())
print("\nMost Frequent Stock Codes:")
display(non_positive_prices["StockCode"].value_counts().head(10))
print("\nNegative-Priced Transactions:")
display(non_positive_prices.loc[non_positive_prices["UnitPrice"] < 0])
print("\nSample Zero-Priced Transactions:")
display(non_positive_prices.loc[non_positive_prices["UnitPrice"] == 0].head(10))

Total Records: 2,517
Zero-Priced Records: 2,515
Negative-Priced Records: 2
Missing Descriptions: 1,454
Missing Customer IDs: 2,477

Quantity Distribution:


count    2,517.00
mean       -53.38
std        539.99
min     -9,600.00
25%        -32.00
50%         -1.00
75%          3.00
max     12,540.00
Name: Quantity, dtype: float64


Most Frequent Stock Codes:


StockCode
23084     16
35965     13
20713     11
22501     10
22676      9
22627      9
22084      9
22683      8
21116      8
46000S     8
Name: count, dtype: int64


Negative-Priced Transactions:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.06",NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.06",NaN,United Kingdom



Sample Zero-Priced Transactions:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.00,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.00,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.00,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.00,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.00,NaN,United Kingdom


**Key Observations**
- **Non-positive unit prices occur in 2,517 records, including 2,515 zero-priced transactions**, with 1,454 missing descriptions and 2,477 missing Customer IDs, indicating a substantial concentration of incomplete transaction information.
- **The only two negative-priced records are explicitly identified as bad debt adjustments of -11,062.06 each**, representing accounting adjustments rather than conventional product sales. Zero-priced transactions contribute no monetary revenue but require appropriate classification during data preparation.

#### Inspecting Duplicate Transactions

In [14]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100
print(f"Duplicate Records: {duplicate_count:,}")
print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")
print("\nSample Duplicate Records:")
display(df.loc[df.duplicated(keep=False)].head(10))

Duplicate Records: 5,268
Duplicate Percentage: 0.97%

Sample Duplicate Records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,"17,908.00",United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,"17,920.00",United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,"17,920.00",United Kingdom


**Key Observations**
- **The dataset contains 5,268 duplicate records (0.97%)**, indicating a relatively small proportion of repeated transaction entries that could influence aggregated daily revenue.
- **Identical product entries appear multiple times within the same invoices**, suggesting either legitimate repeated purchases or duplicated records. Further validation is necessary before removing them to avoid understating actual sales revenue.

#### Inspecting Temporal Coverage

In [15]:
start_date = df["InvoiceDate"].min()
end_date = df["InvoiceDate"].max()
date_range = pd.date_range(start=start_date.normalize(), end=end_date.normalize(), freq="D")
observed_dates = df["InvoiceDate"].dt.normalize().nunique()
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"Total Calendar Days: {len(date_range):,}")
print(f"Days with Transactions: {observed_dates:,}")
print(f"Days without Transactions: {len(date_range) - observed_dates:,}")
print(f"Temporal Coverage: {(observed_dates / len(date_range)) * 100:.2f}%")

Start Date: 2010-12-01 08:26:00
End Date: 2011-12-09 12:50:00
Total Calendar Days: 374
Days with Transactions: 305
Days without Transactions: 69
Temporal Coverage: 81.55%


**Key Observations**
- **The dataset spans 374 calendar days, from December 1, 2010, to December 9, 2011**, providing approximately one year of historical transaction data for time series forecasting.
- **Transactions are recorded on 305 days (81.55% temporal coverage)**, while 69 calendar days (18.45%) contain no recorded transactions, indicating an irregular daily transaction history.
- **The limited historical duration may constrain the reliable identification of annual seasonal patterns**, making weekly seasonality and shorter-term temporal relationships particularly important to investigate.

#### Investigating Missing Transaction Dates by Weekday

In [16]:
transaction_dates = pd.DatetimeIndex(df["InvoiceDate"].dt.normalize().unique())
missing_dates = date_range.difference(transaction_dates)
missing_weekdays = pd.Series(missing_dates.day_name()).value_counts()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
missing_date_summary = pd.DataFrame({
    "Missing_Days": missing_weekdays.reindex(weekday_order, fill_value=0),
    "Total_Calendar_Days": pd.Series(date_range.day_name()).value_counts().reindex(weekday_order, fill_value=0)
})
missing_date_summary["Missing_Percentage"] = (
    missing_date_summary["Missing_Days"] / missing_date_summary["Total_Calendar_Days"] * 100
).round(2)
display(missing_date_summary)
print("\nFirst 20 Missing Dates:")
display(pd.DataFrame({
    "Date": missing_dates[:20],
    "Weekday": missing_dates[:20].day_name()
}))

,Missing_Days,Total_Calendar_Days,Missing_Percentage
Monday,6,53,11.32
Tuesday,1,53,1.89
Wednesday,1,54,1.85
Thursday,1,54,1.85
Friday,4,54,7.41
Saturday,53,53,100.00
Sunday,3,53,5.66



First 20 Missing Dates:


,Date,Weekday
0,2010-12-04,Saturday
1,2010-12-11,Saturday
2,2010-12-18,Saturday
3,2010-12-24,Friday
4,2010-12-25,Saturday
5,2010-12-26,Sunday
6,2010-12-27,Monday
7,2010-12-28,Tuesday
8,2010-12-29,Wednesday
9,2010-12-30,Thursday


**Key Observations**
- **All 53 Saturdays have no recorded transactions (100% missing)**, indicating a consistent weekly non-trading pattern in the dataset rather than randomly distributed missing dates.
- **The remaining 16 missing days occur across other weekdays**, with Mondays accounting for 6 days and Fridays for 4 days, while Tuesdays, Wednesdays, and Thursdays each have only one missing day.
- **A consecutive period without recorded transactions extends from December 24, 2010, through January 3, 2011**, suggesting a possible holiday-related closure or interruption in transaction recording that requires further consideration.
- **The systematic absence of Saturday transactions indicates strong calendar-related structure**, which must be accounted for when constructing the daily revenue series and evaluating forecasting performance.

#### Inspecting Monthly Transaction Distribution

In [17]:
monthly_transactions = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Active_Days=("InvoiceDate", lambda x: x.dt.normalize().nunique())
    )
    .reset_index()
)
monthly_transactions["Month"] = monthly_transactions["InvoiceDate"].astype(str)
monthly_transactions = monthly_transactions[
    ["Month", "Transaction_Records", "Unique_Invoices", "Active_Days"]
]
display(monthly_transactions)

,Month,Transaction_Records,Unique_Invoices,Active_Days
0,2010-12,42481,2025,20
1,2011-01,35147,1476,24
2,2011-02,27707,1393,24
3,2011-03,36748,1983,27
4,2011-04,29916,1744,21
5,2011-05,37030,2162,25
6,2011-06,36874,2012,26
7,2011-07,39518,1927,26
8,2011-08,35284,1737,26
9,2011-09,50226,2327,26


**Key Observations**
- **November 2011 recorded the highest transaction activity, with 84,711 records and 3,462 unique invoices**, followed by October (60,742 records) and September (50,226 records), indicating a substantial increase in retail activity toward the end of the year.
- **February 2011 recorded the lowest transaction volume among the complete months, with 27,707 records**, while monthly activity generally increased during the second half of 2011, suggesting potential growth and seasonal influences that require further investigation.
- **Active trading days varied between 21 and 27 across complete months**, making daily-normalized comparisons important for distinguishing genuine changes in transaction activity from differences in operating days.
- **December 2011 contains only 8 active days and represents an incomplete reporting period**, so its transaction totals should not be directly compared with full months. December 2010 also requires consideration of its holiday-related gaps.

#### Inspecting Geographical Transaction Distribution

In [18]:
country_summary = (
    df.groupby("Country")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Unique_Customers=("CustomerID", "nunique")
    )
    .sort_values("Transaction_Records", ascending=False)
)
country_summary["Transaction_Percentage"] = (
    country_summary["Transaction_Records"] / len(df) * 100
).round(2)
print(f"Total Countries: {df['Country'].nunique()}")
display(country_summary)

Total Countries: 38


,Transaction_Records,Unique_Invoices,Unique_Customers,Transaction_Percentage
Country,,,,
United Kingdom,495478,23494,3950,91.43
Germany,9495,603,95,1.75
France,8557,461,87,1.58
EIRE,8196,360,3,1.51
Spain,2533,105,31,0.47
Netherlands,2371,101,9,0.44
Belgium,2069,119,25,0.38
Switzerland,2002,74,21,0.37
Portugal,1519,71,19,0.28


**Key Observations**
- **The dataset covers 38 geographical markets, with the United Kingdom accounting for 91.43% of all transaction records (495,478)**, demonstrating a substantial concentration of retail activity in the domestic market.
- **Germany (1.75%), France (1.58%), and EIRE (1.51%) represent the largest international markets by transaction volume**, although their individual contributions remain relatively small compared with the United Kingdom.
- **The United Kingdom also has the largest customer base, with 3,950 unique identified customers and 23,494 unique invoices**, reinforcing its dominant position in the dataset.
- **Several international markets contain relatively few transactions and customers**, suggesting that country-level forecasting may be unreliable for smaller markets without sufficient historical observations.

#### Inspecting Product and Invoice Characteristics

In [19]:
product_invoice_summary = pd.DataFrame({
    "Metric": [
        "Unique Stock Codes",
        "Unique Product Descriptions",
        "Unique Invoices",
        "Unique Customers",
        "Average Records per Invoice",
        "Median Records per Invoice",
        "Maximum Records per Invoice"
    ],
    "Value": [
        df["StockCode"].nunique(),
        df["Description"].nunique(),
        df["InvoiceNo"].nunique(),
        df["CustomerID"].nunique(),
        df.groupby("InvoiceNo").size().mean(),
        df.groupby("InvoiceNo").size().median(),
        df.groupby("InvoiceNo").size().max()
    ]
})
display(product_invoice_summary)

,Metric,Value
0,Unique Stock Codes,"4,070.00"
1,Unique Product Descriptions,"4,223.00"
2,Unique Invoices,"25,900.00"
3,Unique Customers,"4,372.00"
4,Average Records per Invoice,20.92
5,Median Records per Invoice,10.00
6,Maximum Records per Invoice,"1,114.00"


**Key Observations**
- **The dataset contains 4,070 unique stock codes and 4,223 unique product descriptions**, indicating substantial product diversity. The difference between these counts warrants further investigation into product naming consistency and special transaction codes.
- **A total of 25,900 unique invoices and 4,372 identified customers are recorded**, providing a broad transaction history, although missing Customer IDs mean the customer count may not represent the complete customer base.
- **Invoices contain an average of 20.92 records compared with a median of 10**, indicating a right-skewed distribution of invoice sizes, with larger invoices influencing the average.
- **The largest invoice contains 1,114 transaction records**, highlighting exceptionally large invoices that may contribute disproportionately to transaction volume and warrant further investigation.

#### Investigating Invoice Size Distribution

In [20]:
invoice_sizes = df.groupby("InvoiceNo").size()
invoice_size_summary = invoice_sizes.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame(name="Transaction_Records")
print("Invoice Size Distribution:")
display(invoice_size_summary)
print("\nTop 10 Largest Invoices:")
display(
    invoice_sizes.nlargest(10)
    .rename("Transaction_Records")
    .reset_index()
)
print(f"\nInvoices with more than 100 records: {(invoice_sizes > 100).sum():,}")
print(f"Percentage of invoices with more than 100 records: {(invoice_sizes > 100).mean() * 100:.2f}%")

Invoice Size Distribution:


,Transaction_Records
count,"25,900.00"
mean,20.92
std,43.03
min,1.00
25%,2.00
50%,10.00
75%,23.00
90%,47.00
95%,68.00
99%,192.01



Top 10 Largest Invoices:


,InvoiceNo,Transaction_Records
0,573585,1114
1,581219,749
2,581492,731
3,580729,721
4,558475,705
5,579777,687
6,581217,676
7,537434,675
8,580730,662
9,538071,652



Invoices with more than 100 records: 667
Percentage of invoices with more than 100 records: 2.58%


**Key Observations**
- **Invoice sizes exhibit a strongly right-skewed distribution, with a median of 10 records compared with an average of 20.92**, indicating that larger invoices substantially influence the overall average.
- **Half of all invoices contain between 2 and 23 transaction records**, while the 90th and 95th percentiles reach 47 and 68 records, respectively, demonstrating considerable variation in invoice complexity.
- **Only 667 invoices (2.58%) contain more than 100 records**, confirming that exceptionally large invoices represent a relatively small proportion of the dataset.
- **The largest invoice (573585) contains 1,114 records, while the 99th percentile is approximately 192 records**, highlighting an extreme upper tail that warrants investigation when evaluating unusual transaction activity and daily revenue fluctuations.

#### Investigating Special Stock Codes

In [21]:
stock_codes = df["StockCode"].astype(str).str.strip()
nonstandard_stock_mask = ~stock_codes.str.fullmatch(r"\d{5}[A-Za-z]?")
special_stock_summary = (
    df.loc[nonstandard_stock_mask]
    .groupby("StockCode")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Unique_Descriptions=("Description", "nunique"),
        Total_Quantity=("Quantity", "sum")
    )
    .sort_values("Transaction_Records", ascending=False)
)
print(f"Records with nonstandard stock codes: {nonstandard_stock_mask.sum():,}")
print(f"Percentage of total records: {nonstandard_stock_mask.mean() * 100:.2f}%")
print(f"Unique nonstandard stock codes: {df.loc[nonstandard_stock_mask, 'StockCode'].nunique():,}")
display(special_stock_summary)
print("\nMost Frequent Descriptions for Nonstandard Stock Codes:")
display(
    df.loc[nonstandard_stock_mask]
    .groupby(["StockCode", "Description"], dropna=False)
    .size()
    .rename("Transaction_Records")
    .sort_values(ascending=False)
    .head(25)
    .reset_index()
)

Records with nonstandard stock codes: 3,385
Percentage of total records: 0.62%
Unique nonstandard stock codes: 37


,Transaction_Records,Unique_Invoices,Unique_Descriptions,Total_Quantity
StockCode,,,,
POST,1256,1254,1,6353
DOT,710,710,1,1707
M,571,517,1,3164
15056BL,326,326,1,2714
C2,144,144,1,290
D,77,65,1,-1194
S,63,19,1,-59
15056bl,62,62,1,87
BANK CHARGES,37,36,1,-13



Most Frequent Descriptions for Nonstandard Stock Codes:


,StockCode,Description,Transaction_Records
0,POST,POSTAGE,1252
1,DOT,DOTCOM POSTAGE,709
2,M,Manual,571
3,15056BL,EDWARDIAN PARASOL BLACK,326
4,C2,CARRIAGE,143
5,D,Discount,77
6,S,SAMPLES,63
7,15056bl,EDWARDIAN PARASOL BLACK,62
8,BANK CHARGES,Bank Charges,37
9,AMAZONFEE,AMAZON FEE,34


**Key Observations**
- **A total of 3,385 transaction records (0.62%) contain nonstandard stock codes, representing 37 unique codes**, indicating that most records follow the conventional product-code format.
- **POST (1,256 records), DOT (710), and M (571) are the most frequent nonstandard codes**, with descriptions identifying postage charges and manual transaction entries.
- **Codes such as C2, D, BANK CHARGES, AMAZONFEE, CRUK, and B represent carriage charges, discounts, fees, commissions, or accounting adjustments**, demonstrating that not every transaction record represents merchandise revenue.
- **Several nonstandard codes correspond to identifiable products, including 15056BL, 15056bl, and DCGS product codes**, confirming that stock-code formatting alone cannot determine whether a record represents a valid merchandise sale.
- **Gift voucher codes and manual transaction entries require further examination**, as their inclusion in the forecasting target depends on the revenue definition and transaction characteristics.

#### Investigating Pricing and Quantity Characteristics of Special Stock Codes

In [22]:
special_transactions = df.loc[nonstandard_stock_mask].copy()
special_transactions["Line_Amount"] = (
    special_transactions["Quantity"] * special_transactions["UnitPrice"]
)
special_pricing_summary = (
    special_transactions.groupby("StockCode")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Negative_Quantity_Records=("Quantity", lambda x: (x < 0).sum()),
        Zero_Price_Records=("UnitPrice", lambda x: (x == 0).sum()),
        Negative_Price_Records=("UnitPrice", lambda x: (x < 0).sum()),
        Min_Unit_Price=("UnitPrice", "min"),
        Max_Unit_Price=("UnitPrice", "max"),
        Net_Line_Amount=("Line_Amount", "sum")
    )
    .sort_values("Transaction_Records", ascending=False)
)
display(special_pricing_summary)

,Transaction_Records,Negative_Quantity_Records,Zero_Price_Records,Negative_Price_Records,Min_Unit_Price,Max_Unit_Price,Net_Line_Amount
StockCode,,,,,,,
POST,1256,126,4,0,0.00,"8,142.75","66,230.64"
DOT,710,1,3,0,0.00,"4,505.17","206,245.48"
M,571,244,6,0,0.00,"38,970.00","-68,674.19"
15056BL,326,5,0,0,1.95,12.72,"15,176.34"
C2,144,2,1,0,0.00,150.00,"6,986.00"
D,77,77,0,0,0.01,"1,867.86","-5,696.22"
S,63,61,0,0,2.80,570.00,"-3,049.39"
15056bl,62,0,0,0,12.46,12.72,"1,086.88"
BANK CHARGES,37,25,0,0,0.00,"1,050.15","-7,175.64"


**Key Observations**
- **Postage and carriage codes contribute substantial positive net amounts**, including DOT (206,245.48), POST (66,230.64), and C2 (6,986.00). These represent additional transaction charges rather than conventional merchandise sales and require separate treatment when defining the forecasting target.
- **AMAZONFEE (-221,520.50), CRUK (-7,933.43), BANK CHARGES (-7,175.64), and D (-5,696.22) contribute negative net amounts**, demonstrating how fees, commissions, and discounts can affect revenue calculations if all transaction records are aggregated indiscriminately.
- **Manual transactions (M) contain both positive and negative quantities, with unit prices reaching 38,970.00 and a net amount of -68,674.19**, indicating substantial financial variability that warrants further investigation before inclusion in sales forecasting.
- **Several nonstandard stock codes represent legitimate merchandise with positive transaction amounts**, including 15056BL, 15056bl, DCGSSGIRL, and DCGSSBOY, reinforcing that stock-code formatting alone is not a reliable exclusion criterion.
- **Gift voucher entries, bad-debt adjustments, and zero-priced records require distinct consideration**, as their financial treatment differs from ordinary product sales and may otherwise distort the daily revenue series.

#### Investigating Transaction Activity by Hour

In [23]:
hourly_transaction_summary = (
    df.groupby(df["InvoiceDate"].dt.hour)
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique")
    )
    .reindex(range(24), fill_value=0)
)
hourly_transaction_summary.index.name = "Hour"
hourly_transaction_summary["Transaction_Percentage"] = (
    hourly_transaction_summary["Transaction_Records"] / len(df) * 100
).round(2)
display(hourly_transaction_summary)
print(f"Peak Transaction Hour: {hourly_transaction_summary['Transaction_Records'].idxmax():02d}:00")
print(f"Hours with Recorded Transactions: {(hourly_transaction_summary['Transaction_Records'] > 0).sum()}")

,Transaction_Records,Unique_Invoices,Transaction_Percentage
Hour,,,
0,0,0,0.00
1,0,0,0.00
2,0,0,0.00
3,0,0,0.00
4,0,0,0.00
5,0,0,0.00
6,41,22,0.01
7,383,31,0.07
8,8909,624,1.64


Peak Transaction Hour: 12:00
Hours with Recorded Transactions: 15


**Key Observations**
- **Transaction activity is concentrated between 09:00 and 17:00, accounting for approximately 95.95% of all transaction records**, indicating a strong daytime operating pattern.
- **The highest transaction volume occurs at 12:00 (78,709 records; 14.52%)**, closely followed by 15:00 (77,519 records; 14.30%), identifying midday and mid-afternoon as the busiest recorded periods.
- **No transactions are recorded between 21:00 and 05:59**, while activity before 08:00 and after 19:00 is minimal, suggesting relatively consistent transaction recording hours.
- **Hourly transaction volumes and unique invoice counts do not always follow the same pattern**, indicating that differences in invoice sizes also influence the number of records observed during each hour.

#### Investigating Transaction Activity by Weekday

In [24]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]
weekday_transaction_summary = (
    df.groupby(df["InvoiceDate"].dt.day_name())
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Active_Days=("InvoiceDate", lambda x: x.dt.normalize().nunique())
    )
    .reindex(weekday_order, fill_value=0)
)
weekday_transaction_summary["Records_Per_Active_Day"] = (
    weekday_transaction_summary["Transaction_Records"]
    / weekday_transaction_summary["Active_Days"].replace(0, np.nan)
).round(2)
weekday_transaction_summary["Transaction_Percentage"] = (
    weekday_transaction_summary["Transaction_Records"] / len(df) * 100
).round(2)
display(weekday_transaction_summary)

,Transaction_Records,Unique_Invoices,Active_Days,Records_Per_Active_Day,Transaction_Percentage
InvoiceDate,,,,,
Monday,95111,4138,47,"2,023.64",17.55
Tuesday,101808,4722,52,"1,957.85",18.79
Wednesday,94565,4815,53,"1,784.25",17.45
Thursday,103857,5660,53,"1,959.57",19.17
Friday,82193,4184,50,"1,643.86",15.17
Saturday,0,0,0,NaN,0.00
Sunday,64375,2381,50,"1,287.50",11.88


**Key Observations**
- **Thursday records the highest overall transaction volume (103,857 records; 19.17%) and the most unique invoices (5,660)**, followed by Tuesday with 101,808 transaction records (18.79%).
- **Monday has the highest average transaction activity per active day (2,023.64 records)**, followed by Thursday (1,959.57) and Tuesday (1,957.85), demonstrating that total transaction volume alone does not fully capture weekday activity patterns.
- **Sunday has the lowest transaction activity among operating weekdays, averaging 1,287.50 records per active day**, while Friday also exhibits comparatively lower activity at 1,643.86 records per active day.
- **Saturday has no recorded transactions across the entire observation period**, confirming a consistent weekly non-trading pattern that must be considered when constructing the daily revenue series and designing forecasting models.

#### Investigating Transaction Activity by Month and Weekday

In [25]:
monthly_weekday_summary = (
    df.assign(
        Month=df["InvoiceDate"].dt.to_period("M").astype(str),
        Weekday=df["InvoiceDate"].dt.day_name(),
        Transaction_Date=df["InvoiceDate"].dt.normalize()
    )
    .groupby(["Month", "Weekday"])
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Active_Days=("Transaction_Date", "nunique")
    )
)
monthly_weekday_summary["Records_Per_Active_Day"] = (
    monthly_weekday_summary["Transaction_Records"]
    / monthly_weekday_summary["Active_Days"]
)
monthly_weekday_pivot = (
    monthly_weekday_summary["Records_Per_Active_Day"]
    .unstack("Weekday")
    .reindex(columns=weekday_order)
    .round(2)
)
display(monthly_weekday_pivot)

Weekday,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Month,,,,,,,
2010-12,"2,641.33","2,212.00","1,848.75","1,938.25","2,691.67",NaN,"1,566.00"
2011-01,"1,881.75","1,434.75","1,564.25","1,588.50","1,476.50",NaN,841.00
2011-02,"1,382.75","1,432.50","1,264.75","1,196.75","1,005.25",NaN,644.75
2011-03,"1,455.50","1,724.00","1,199.00","1,581.80","1,207.75",NaN,892.75
2011-04,"2,057.33","1,539.25","1,126.50","1,564.25","1,395.00",NaN,879.67
2011-05,"1,673.67","1,799.60","1,387.50","1,680.75","1,443.75",NaN,992.60
2011-06,"1,475.75","1,523.50","1,575.20","1,713.00",981.25,NaN,"1,127.75"
2011-07,"1,973.75","1,703.75","1,677.00","1,702.25","1,245.00",NaN,"1,013.20"
2011-08,"1,222.00","1,643.20","1,556.60","1,676.00","1,081.25",NaN,842.00


**Key Observations**
- **Average daily transaction activity increases substantially across most weekdays during September–November 2011**, with November recording particularly high activity across all operating weekdays, suggesting a broad increase in transaction demand rather than growth concentrated on a single day.
- **Weekday activity patterns vary across months, with no single weekday consistently recording the highest average transaction volume**, indicating that weekly transaction behavior changes over time.
- **Sunday generally records lower average transaction activity than other operating weekdays**, although its activity increases considerably during November 2011, reaching 3,083 records per active day.
- **Saturday remains entirely absent throughout the observation period**, reinforcing the consistent weekly non-trading pattern identified in earlier analyses.
- **December 2011 shows elevated activity on several weekdays, but its incomplete reporting period limits direct comparisons**, particularly because some weekday averages are based on very few observations.

#### Investigating Transaction Frequency by Customer

In [26]:
customer_invoice_counts = (
    df.dropna(subset=["CustomerID"])
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)
customer_frequency_summary = (
    customer_invoice_counts.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_frame(name="Unique_Invoices")
)
print("Customer Invoice Frequency Distribution:")
display(customer_frequency_summary)
print("\nAdditional Customer Frequency Metrics:")
print(f"Customers with exactly one invoice: {(customer_invoice_counts == 1).sum():,}")
print(f"Customers with more than 10 invoices: {(customer_invoice_counts > 10).sum():,}")
print(f"Maximum invoices for a single customer: {customer_invoice_counts.max():,}")

Customer Invoice Frequency Distribution:


,Unique_Invoices
count,"4,372.00"
mean,5.08
std,9.34
min,1.00
25%,1.00
50%,3.00
75%,5.00
90%,11.00
95%,16.00
99%,36.00



Additional Customer Frequency Metrics:
Customers with exactly one invoice: 1,313
Customers with more than 10 invoices: 459
Maximum invoices for a single customer: 248


**Key Observations**
- **The dataset contains 4,372 identifiable customers, with a median of 3 unique invoices per customer compared with an average of 5.08**, indicating a right-skewed distribution of customer transaction frequency.
- **A total of 1,313 customers (30.03%) have exactly one recorded invoice**, while 459 customers (10.50%) have more than 10 invoices, highlighting differences in customer purchasing frequency.
- **The 90th percentile reaches 11 invoices, while the 99th percentile reaches 36 invoices**, demonstrating that a relatively small proportion of customers exhibit substantially higher invoice activity.
- **The most active customer is associated with 248 unique invoices**, indicating an extreme upper tail that may reflect high-frequency or business purchasing behavior.
- **These findings are limited to identifiable customers and include cancellation invoices**, so recorded invoice frequency should not be interpreted directly as completed purchase frequency.

#### Investigating Transaction Activity by Product

In [27]:
product_transaction_counts = (
    df.groupby("StockCode")
    .size()
    .sort_values(ascending=False)
)
product_frequency_summary = (
    product_transaction_counts.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_frame(name="Transaction_Records")
)
print("Product Transaction Frequency Distribution:")
display(product_frequency_summary)
print("\nTop 10 Most Frequently Recorded Stock Codes:")
display(
    product_transaction_counts.head(10)
    .rename("Transaction_Records")
    .reset_index()
)
print("\nAdditional Product Frequency Metrics:")
print(f"Stock codes appearing once: {(product_transaction_counts == 1).sum():,}")
print(f"Stock codes appearing more than 100 times: {(product_transaction_counts > 100).sum():,}")
print(f"Maximum records for a single stock code: {product_transaction_counts.max():,}")

Product Transaction Frequency Distribution:


,Transaction_Records
count,"4,070.00"
mean,133.15
std,199.90
min,1.00
25%,14.00
50%,62.00
75%,167.00
90%,346.10
95%,495.55
99%,988.17



Top 10 Most Frequently Recorded Stock Codes:


,StockCode,Transaction_Records
0,85123A,2313
1,22423,2203
2,85099B,2159
3,47566,1727
4,20725,1639
5,84879,1502
6,22720,1477
7,22197,1476
8,21212,1385
9,20727,1350



Additional Product Frequency Metrics:
Stock codes appearing once: 233
Stock codes appearing more than 100 times: 1,563
Maximum records for a single stock code: 2,313


**Key Observations**
- **Product transaction frequency exhibits a strongly right-skewed distribution, with a median of 62 records compared with an average of 133.15 per stock code**, indicating substantial variation in product-level transaction activity.
- **A total of 233 stock codes (5.72%) appear only once, while 1,563 stock codes (38.40%) appear more than 100 times**, highlighting differences in how frequently products are represented in the dataset.
- **The 90th percentile reaches approximately 346 records, while the 99th percentile reaches 988 records**, demonstrating a pronounced upper tail of frequently recorded products.
- **Stock code 85123A is the most frequently recorded product, appearing 2,313 times**, followed by 22423 (2,203) and 85099B (2,159), indicating substantial transaction activity among the leading stock codes.
- **Transaction frequency does not directly represent sales volume or revenue**, since records may contain different quantities, cancellations, and adjustments. Product-level revenue contributions require separate investigation.

#### Investigating Product Description Consistency

In [28]:
stock_description_summary = (
    df.groupby("StockCode")
    .agg(
        Unique_Descriptions=("Description", "nunique"),
        Missing_Descriptions=("Description", lambda x: x.isna().sum()),
        Transaction_Records=("InvoiceNo", "size")
    )
)
multiple_description_codes = stock_description_summary[
    stock_description_summary["Unique_Descriptions"] > 1
].sort_values(
    ["Unique_Descriptions", "Transaction_Records"],
    ascending=False
)
print(f"Stock codes with multiple descriptions: {len(multiple_description_codes):,}")
print(f"Percentage of stock codes with multiple descriptions: {len(multiple_description_codes) / df['StockCode'].nunique() * 100:.2f}%")
print(f"Maximum descriptions for a single stock code: {stock_description_summary['Unique_Descriptions'].max():,}")
print("\nTop 15 Stock Codes with Multiple Descriptions:")
display(multiple_description_codes.head(15))
print("\nExamples of Description Variations:")
example_codes = multiple_description_codes.head(5).index
display(
    df[df["StockCode"].isin(example_codes)]
    .groupby(["StockCode", "Description"], dropna=False)
    .size()
    .rename("Transaction_Records")
    .reset_index()
    .sort_values(["StockCode", "Transaction_Records"], ascending=[True, False])
)

Stock codes with multiple descriptions: 650
Percentage of stock codes with multiple descriptions: 15.97%
Maximum descriptions for a single stock code: 8

Top 15 Stock Codes with Multiple Descriptions:


,Unique_Descriptions,Missing_Descriptions,Transaction_Records
StockCode,,,
20713,8,4,684
23084,7,10,1067
85175,6,1,154
21830,6,2,112
21181,5,2,882
23343,5,0,501
23131,5,1,122
72807A,5,0,118
85172,5,0,53



Examples of Description Variations:


,StockCode,Description,Transaction_Records
1,20713,JUMBO BAG OWLS,673
8,20713,NaN,4
0,20713,Found,1
2,20713,Marked as 23343,1
3,20713,found,1
4,20713,wrongly coded 23343,1
5,20713,wrongly coded-23343,1
6,20713,wrongly marked 23343,1
7,20713,wrongly marked. 23343 in box,1
9,21181,PLEASE ONE PERSON METAL SIGN,875


**Key Observations**
- **A total of 650 stock codes (15.97%) have multiple distinct descriptions**, with a maximum of 8 descriptions associated with a single stock code, indicating inconsistencies in product-description records.
- **Most examined stock codes have one dominant product description**, such as JUMBO BAG OWLS (673 of 684 records for stock code 20713) and RABBIT NIGHT LIGHT (1,051 of 1,067 records for stock code 23084), while alternative descriptions occur relatively infrequently.
- **Several alternative descriptions contain administrative notes, inventory adjustments, stock corrections, or operational comments**, including entries such as "wrongly coded," "adjustment," "damaged," and "allocate stock," suggesting that some records may not represent conventional customer purchases.
- **Missing descriptions also occur within stock codes that otherwise have identifiable products**, demonstrating that description completeness and consistency require consideration during data preparation.
- **Stock codes alone may not reliably distinguish merchandise sales from operational adjustments**, making transaction quantities, prices, descriptions, and invoice characteristics important for defining valid sales revenue.

#### Investigating Missing Product Description Characteristics

In [29]:
missing_description_records = df[df["Description"].isna()].copy()
missing_description_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Unique Stock Codes",
        "Unique Invoices",
        "Missing Customer IDs",
        "Negative Quantity Records",
        "Positive Quantity Records",
        "Zero Price Records",
        "Positive Price Records",
        "Negative Price Records",
        "Cancellation Invoice Records"
    ],
    "Value": [
        len(missing_description_records),
        missing_description_records["StockCode"].nunique(),
        missing_description_records["InvoiceNo"].nunique(),
        missing_description_records["CustomerID"].isna().sum(),
        (missing_description_records["Quantity"] < 0).sum(),
        (missing_description_records["Quantity"] > 0).sum(),
        (missing_description_records["UnitPrice"] == 0).sum(),
        (missing_description_records["UnitPrice"] > 0).sum(),
        (missing_description_records["UnitPrice"] < 0).sum(),
        missing_description_records["InvoiceNo"].astype(str).str.startswith("C").sum()
    ]
})
display(missing_description_summary)

,Metric,Value
0,Total Records,1454
1,Unique Stock Codes,960
2,Unique Invoices,1454
3,Missing Customer IDs,1454
4,Negative Quantity Records,862
5,Positive Quantity Records,592
6,Zero Price Records,1454
7,Positive Price Records,0
8,Negative Price Records,0
9,Cancellation Invoice Records,0


**Key Observations**
- **All 1,454 records with missing product descriptions also have missing Customer IDs and zero unit prices**, indicating a consistent data-quality pattern rather than isolated missing values.
- **These records span 960 unique stock codes and 1,454 unique invoices**, demonstrating that missing descriptions occur across a broad range of products and transaction identifiers.
- **Negative quantities account for 862 records (59.28%), while 592 records (40.72%) have positive quantities**, suggesting that these entries may include inventory movements or administrative adjustments rather than conventional customer purchases.
- **None of the records have positive or negative unit prices, and none belong to cancellation-prefixed invoices**, confirming that they contribute zero monetary value under the Quantity × UnitPrice calculation.
- **Missing-description records can be excluded from revenue aggregation without directly changing calculated revenue totals**, although their potential operational significance should be distinguished from valid sales activity during data preparation.

#### Investigating Customer Identification Across Transactions

In [30]:
customer_id_summary = (
    df.assign(
        Customer_ID_Status=np.where(
            df["CustomerID"].isna(),
            "Missing Customer ID",
            "Identified Customer"
        ),
        Line_Amount=df["Quantity"] * df["UnitPrice"]
    )
    .groupby("Customer_ID_Status")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Unique_Stock_Codes=("StockCode", "nunique"),
        Positive_Quantity_Records=("Quantity", lambda x: (x > 0).sum()),
        Negative_Quantity_Records=("Quantity", lambda x: (x < 0).sum()),
        Positive_Price_Records=("UnitPrice", lambda x: (x > 0).sum()),
        Zero_Price_Records=("UnitPrice", lambda x: (x == 0).sum()),
        Net_Line_Amount=("Line_Amount", "sum")
    )
)
customer_id_summary["Transaction_Percentage"] = (
    customer_id_summary["Transaction_Records"] / len(df) * 100
).round(2)
display(customer_id_summary)

,Transaction_Records,Unique_Invoices,Unique_Stock_Codes,Positive_Quantity_Records,Negative_Quantity_Records,Positive_Price_Records,Zero_Price_Records,Net_Line_Amount,Transaction_Percentage
Customer_ID_Status,,,,,,,,,
Identified Customer,406829,22190,3684,397924,8905,406789,40,"8,300,065.81",75.07
Missing Customer ID,135080,3710,3810,133361,1719,132603,2475,"1,447,682.12",24.93


**Key Observations**
- **Missing Customer IDs affect 135,080 transaction records (24.93%) across 3,710 unique invoices**, representing a substantial portion of the dataset rather than an isolated data-quality issue.
- **Transactions with missing Customer IDs contribute 1,447,682.12 in net line amounts, approximately 14.85% of the dataset's total calculated net amount**, demonstrating that excluding these records would materially affect aggregate revenue calculations.
- **Among records with missing Customer IDs, 133,361 have positive quantities and 132,603 have positive unit prices**, indicating that many contain characteristics consistent with ordinary sales transactions.
- **Identified-customer transactions contribute 8,300,065.81 in net line amounts across 22,190 invoices**, accounting for the majority of recorded financial activity.
- **Customer ID availability alone is not a suitable criterion for excluding transactions from aggregate daily revenue forecasting**, although transaction validity must still be assessed using invoice status, quantities, prices, and special transaction codes.

#### Investigating Cancellation Transaction Characteristics

In [31]:
cancellation_mask = df["InvoiceNo"].astype(str).str.startswith("C")
cancellation_records = df.loc[cancellation_mask].copy()
cancellation_records["Line_Amount"] = (
    cancellation_records["Quantity"] * cancellation_records["UnitPrice"]
)
cancellation_summary = pd.DataFrame({
    "Metric": [
        "Cancellation Records",
        "Unique Cancellation Invoices",
        "Unique Stock Codes",
        "Unique Identified Customers",
        "Missing Customer IDs",
        "Negative Quantity Records",
        "Positive Quantity Records",
        "Zero Price Records",
        "Positive Price Records",
        "Total Cancelled Units",
        "Net Cancellation Amount"
    ],
    "Value": [
        len(cancellation_records),
        cancellation_records["InvoiceNo"].nunique(),
        cancellation_records["StockCode"].nunique(),
        cancellation_records["CustomerID"].nunique(),
        cancellation_records["CustomerID"].isna().sum(),
        (cancellation_records["Quantity"] < 0).sum(),
        (cancellation_records["Quantity"] > 0).sum(),
        (cancellation_records["UnitPrice"] == 0).sum(),
        (cancellation_records["UnitPrice"] > 0).sum(),
        cancellation_records["Quantity"].sum(),
        cancellation_records["Line_Amount"].sum()
    ]
})
display(cancellation_summary)

,Metric,Value
0,Cancellation Records,"9,288.00"
1,Unique Cancellation Invoices,"3,836.00"
2,Unique Stock Codes,"1,947.00"
3,Unique Identified Customers,"1,589.00"
4,Missing Customer IDs,383.00
5,Negative Quantity Records,"9,288.00"
6,Positive Quantity Records,0.00
7,Zero Price Records,0.00
8,Positive Price Records,"9,288.00"
9,Total Cancelled Units,"-277,574.00"


**Key Observations**
- **The dataset contains 9,288 cancellation records across 3,836 unique cancellation invoices**, representing approximately 1.71% of all transaction records.
- **All cancellation records have negative quantities and positive unit prices**, confirming a consistent cancellation structure, with a combined quantity of -277,574 units.
- **Cancellation-prefixed transactions contribute -896,812.49 in net line amounts**, demonstrating their substantial financial impact and the importance of distinguishing gross sales from sales after cancellations.
- **Cancellations span 1,947 unique stock codes and 1,589 identified customers**, indicating that cancellation activity affects a broad range of products and customers rather than a small isolated group.
- **Only 383 cancellation records have missing Customer IDs**, suggesting that most cancellation activity can be associated with identified customers, although customer identification is not required for aggregate revenue calculations.

#### Investigating the Financial Impact of Transaction Types

In [32]:
transaction_type_summary = (
    df.assign(
        Transaction_Type=np.select(
            [
                df["InvoiceNo"].astype(str).str.startswith("C"),
                df["Quantity"] < 0,
                df["UnitPrice"] < 0,
                df["UnitPrice"] == 0
            ],
            [
                "Cancellation Invoice",
                "Other Negative Quantity",
                "Negative Unit Price",
                "Zero Unit Price"
            ],
            default="Positive Quantity and Price"
        ),
        Line_Amount=df["Quantity"] * df["UnitPrice"]
    )
    .groupby("Transaction_Type")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Total_Quantity=("Quantity", "sum"),
        Net_Line_Amount=("Line_Amount", "sum")
    )
)
transaction_type_summary["Transaction_Percentage"] = (
    transaction_type_summary["Transaction_Records"] / len(df) * 100
).round(2)
display(transaction_type_summary)
print(f"\nTotal Calculated Net Line Amount: {transaction_type_summary['Net_Line_Amount'].sum():,.2f}")

,Transaction_Records,Unique_Invoices,Total_Quantity,Net_Line_Amount,Transaction_Percentage
Transaction_Type,,,,,
Cancellation Invoice,9288,3836,-277574,"-896,812.49",1.71
Negative Unit Price,2,2,2,"-22,124.12",0.00
Other Negative Quantity,1336,1336,-206957,0.00,0.25
Positive Quantity and Price,530104,19960,5588376,"10,666,684.54",97.82
Zero Unit Price,1179,819,72603,0.00,0.22



Total Calculated Net Line Amount: 9,747,747.93


**Key Observations**
- **Positive-quantity, positive-price transactions account for 530,104 records (97.82%) and contribute 10,666,684.54 in calculated line amounts**, representing the primary source of positive financial activity in the dataset.
- **Cancellation invoices contribute -896,812.49 across 9,288 records (1.71%)**, demonstrating their substantial impact on the difference between positive transaction amounts and overall net transaction value.
- **Two negative-price records contribute an additional -22,124.12**, corresponding to previously identified bad-debt adjustments rather than conventional merchandise sales.
- **Other negative-quantity records (1,336) and zero-price records (1,179) contribute no monetary value**, although their quantities and transaction characteristics remain relevant to understanding data quality and operational adjustments.
- **The total calculated net line amount is 9,747,747.93**, but this remains a preliminary financial measure because it includes postage, fees, discounts, vouchers, and other non-merchandise entries.

#### Investigating Daily Transaction Coverage and Activity

In [33]:
daily_transaction_summary = (
    df.assign(Transaction_Date=df["InvoiceDate"].dt.normalize())
    .groupby("Transaction_Date")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Unique_Stock_Codes=("StockCode", "nunique")
    )
    .reindex(date_range)
)
daily_transaction_summary.index.name = "Date"
daily_activity_summary = (
    daily_transaction_summary["Transaction_Records"]
    .dropna()
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame(name="Transaction_Records")
)
print("Daily Transaction Activity on Recorded Dates:")
display(daily_activity_summary)
print("\nCalendar Coverage:")
print(f"Total calendar days: {len(daily_transaction_summary):,}")
print(f"Days with recorded transactions: {daily_transaction_summary['Transaction_Records'].notna().sum():,}")
print(f"Days without recorded transactions: {daily_transaction_summary['Transaction_Records'].isna().sum():,}")
print("\nTop 10 Dates by Transaction Records:")
display(
    daily_transaction_summary.nlargest(10, "Transaction_Records")
    .reset_index()[["Date", "Transaction_Records", "Unique_Invoices", "Unique_Stock_Codes"]]
)

Daily Transaction Activity on Recorded Dates:


,Transaction_Records
count,305.00
mean,"1,776.75"
std,829.89
min,279.00
25%,"1,229.00"
50%,"1,577.00"
75%,"2,109.00"
90%,"2,970.80"
95%,"3,404.20"
99%,"4,190.76"



Calendar Coverage:
Total calendar days: 374
Days with recorded transactions: 305
Days without recorded transactions: 69

Top 10 Dates by Transaction Records:


,Date,Transaction_Records,Unique_Invoices,Unique_Stock_Codes
0,2011-12-05,"5,331.00",151.00,"1,774.00"
1,2011-12-08,"4,940.00",145.00,"1,677.00"
2,2011-11-29,"4,313.00",151.00,"1,560.00"
3,2011-11-16,"4,195.00",161.00,"1,537.00"
4,2011-11-11,"4,089.00",118.00,"1,536.00"
5,2011-11-08,"4,070.00",144.00,"1,487.00"
6,2011-11-22,"3,967.00",174.00,"1,527.00"
7,2010-12-06,"3,878.00",133.00,"1,479.00"
8,2011-11-24,"3,766.00",146.00,"1,528.00"
9,2011-11-17,"3,621.00",170.00,"1,441.00"


**Key Observations**
- **Daily transaction activity averages 1,776.75 records across 305 observed trading days, with a median of 1,577 records**, indicating a right-skewed distribution influenced by high-activity days.
- **The middle 50% of observed days contain between 1,229 and 2,109 transaction records**, while the 95th and 99th percentiles reach approximately 3,404 and 4,191 records, respectively, demonstrating considerable variation in daily activity.
- **December 5, 2011 records the highest transaction volume (5,331 records)**, followed by December 8 (4,940 records), with most of the ten busiest dates occurring during November and early December 2011.
- **The observation period covers 374 calendar days, of which 305 contain recorded transactions and 69 have no recorded activity**, reinforcing the need to distinguish scheduled non-trading days from possible reporting gaps.
- **High transaction-record counts do not necessarily imply equally high sales revenue**, since invoice sizes, quantities, prices, cancellations, and non-merchandise entries can influence the relationship between transaction activity and financial value.

#### Investigating the Financial Impact of Duplicate Records

In [34]:
duplicate_mask = df.duplicated(keep="first")
duplicate_records = df.loc[duplicate_mask].copy()
duplicate_records["Line_Amount"] = (
    duplicate_records["Quantity"] * duplicate_records["UnitPrice"]
)
duplicate_summary = pd.DataFrame({
    "Metric": [
        "Duplicate Records",
        "Duplicate Percentage",
        "Unique Invoices Affected",
        "Unique Stock Codes Affected",
        "Positive Quantity Records",
        "Negative Quantity Records",
        "Zero Price Records",
        "Duplicate Line Amount",
        "Positive Duplicate Line Amount",
        "Negative Duplicate Line Amount"
    ],
    "Value": [
        duplicate_mask.sum(),
        duplicate_mask.mean() * 100,
        duplicate_records["InvoiceNo"].nunique(),
        duplicate_records["StockCode"].nunique(),
        (duplicate_records["Quantity"] > 0).sum(),
        (duplicate_records["Quantity"] < 0).sum(),
        (duplicate_records["UnitPrice"] == 0).sum(),
        duplicate_records["Line_Amount"].sum(),
        duplicate_records.loc[
            duplicate_records["Line_Amount"] > 0, "Line_Amount"
        ].sum(),
        duplicate_records.loc[
            duplicate_records["Line_Amount"] < 0, "Line_Amount"
        ].sum()
    ]
})
display(duplicate_summary)
print("\nTop 10 Invoices by Duplicate Record Count:")
display(
    duplicate_records.groupby("InvoiceNo")
    .size()
    .nlargest(10)
    .rename("Duplicate_Records")
    .reset_index()
)

,Metric,Value
0,Duplicate Records,"5,268.00"
1,Duplicate Percentage,0.97
2,Unique Invoices Affected,"1,933.00"
3,Unique Stock Codes Affected,"1,706.00"
4,Positive Quantity Records,"5,231.00"
5,Negative Quantity Records,37.00
6,Zero Price Records,5.00
7,Duplicate Line Amount,"21,740.98"
8,Positive Duplicate Line Amount,"24,573.74"
9,Negative Duplicate Line Amount,"-2,832.76"



Top 10 Invoices by Duplicate Record Count:


,InvoiceNo,Duplicate_Records
0,555524,34
1,547651,31
2,579458,28
3,572103,27
4,574481,27
5,577482,26
6,537781,25
7,552285,24
8,569897,21
9,575117,21


**Key Observations**
- **The dataset contains 5,268 exact duplicate records (0.97%), affecting 1,933 unique invoices and 1,706 stock codes**, indicating that repeated identical entries occur across a broad range of transactions.
- **Most duplicate records have positive quantities (5,231), while only 37 have negative quantities and 5 have zero unit prices**, suggesting that duplicates predominantly occur among positive-quantity transaction entries.
- **Duplicate records contribute a net calculated line amount of 21,740.98**, comprising 24,573.74 in positive amounts and -2,832.76 in negative amounts. This represents approximately 0.22% of the dataset's preliminary net transaction value.
- **Invoice 555524 contains the highest number of duplicate records (34)**, followed by invoices 547651 (31) and 579458 (28), demonstrating that repeated entries are more concentrated in certain invoices.
- **Exact duplication does not necessarily indicate erroneous transactions**, as identical product lines may represent legitimate repeated entries. Removing duplicates without further validation could incorrectly reduce calculated sales revenue.

#### Investigating Transaction Date and Invoice Consistency

In [35]:
invoice_consistency = (
    df.groupby("InvoiceNo")
    .agg(
        Transaction_Records=("StockCode", "size"),
        Unique_Timestamps=("InvoiceDate", "nunique"),
        Unique_Dates=("InvoiceDate", lambda x: x.dt.normalize().nunique()),
        Unique_Countries=("Country", "nunique"),
        Unique_Customers=("CustomerID", "nunique"),
        First_Transaction=("InvoiceDate", "min"),
        Last_Transaction=("InvoiceDate", "max")
    )
)
consistency_summary = pd.DataFrame({
    "Metric": [
        "Total Unique Invoices",
        "Invoices with Multiple Timestamps",
        "Invoices Spanning Multiple Dates",
        "Invoices Associated with Multiple Countries",
        "Invoices Associated with Multiple Customers",
        "Maximum Unique Timestamps per Invoice",
        "Maximum Unique Dates per Invoice"
    ],
    "Value": [
        len(invoice_consistency),
        (invoice_consistency["Unique_Timestamps"] > 1).sum(),
        (invoice_consistency["Unique_Dates"] > 1).sum(),
        (invoice_consistency["Unique_Countries"] > 1).sum(),
        (invoice_consistency["Unique_Customers"] > 1).sum(),
        invoice_consistency["Unique_Timestamps"].max(),
        invoice_consistency["Unique_Dates"].max()
    ]
})
display(consistency_summary)
print("\nExamples of Invoices with Multiple Timestamps:")
display(
    invoice_consistency[
        invoice_consistency["Unique_Timestamps"] > 1
    ].head(10)
)

,Metric,Value
0,Total Unique Invoices,25900
1,Invoices with Multiple Timestamps,43
2,Invoices Spanning Multiple Dates,0
3,Invoices Associated with Multiple Countries,0
4,Invoices Associated with Multiple Customers,0
5,Maximum Unique Timestamps per Invoice,2
6,Maximum Unique Dates per Invoice,1



Examples of Invoices with Multiple Timestamps:


,Transaction_Records,Unique_Timestamps,Unique_Dates,Unique_Countries,Unique_Customers,First_Transaction,Last_Transaction
InvoiceNo,,,,,,,
536591,40,2,1,1,1,2010-12-01 16:57:00,2010-12-01 16:58:00
540185,29,2,1,1,1,2011-01-05 13:40:00,2011-01-05 13:41:00
541596,103,2,1,1,1,2011-01-19 16:18:00,2011-01-19 16:19:00
541631,72,2,1,1,1,2011-01-20 10:47:00,2011-01-20 10:48:00
541809,91,2,1,1,0,2011-01-21 14:58:00,2011-01-21 14:59:00
541816,73,2,1,1,1,2011-01-21 15:56:00,2011-01-21 15:57:00
541849,49,2,1,1,1,2011-01-23 13:33:00,2011-01-23 13:34:00
542217,18,2,1,1,1,2011-01-26 12:35:00,2011-01-26 12:36:00
542806,39,2,1,1,1,2011-02-01 11:19:00,2011-02-01 11:20:00


**Key Observations**
- **All 25,900 unique invoices are associated with a single calendar date**, confirming that individual invoices do not span multiple days and can be assigned consistently during daily revenue aggregation.
- **Only 43 invoices (0.17%) contain multiple transaction timestamps**, with a maximum of two distinct timestamps per invoice. The displayed examples show differences of approximately one minute, suggesting minor timestamp variations within the same invoice.
- **No invoices are associated with multiple countries or multiple identified customers**, indicating consistent geographical and customer attribution at the invoice level.
- **The absence of invoices spanning multiple dates reduces the risk of splitting invoice-level transaction amounts across different forecasting dates**, supporting reliable daily aggregation based on the InvoiceDate field.

#### Investigating Daily Revenue Coverage by Transaction Type

In [37]:
revenue_coverage_df = df.assign(
    Transaction_Date=df["InvoiceDate"].dt.normalize(),
    Line_Amount=df["Quantity"] * df["UnitPrice"],
    Is_Cancellation=df["InvoiceNo"].astype(str).str.startswith("C")
)
daily_revenue_coverage = (
    revenue_coverage_df.groupby("Transaction_Date")
    .agg(
        Total_Records=("InvoiceNo", "size"),
        Positive_Amount_Records=("Line_Amount", lambda x: (x > 0).sum()),
        Negative_Amount_Records=("Line_Amount", lambda x: (x < 0).sum()),
        Zero_Amount_Records=("Line_Amount", lambda x: (x == 0).sum()),
        Cancellation_Records=("Is_Cancellation", "sum"),
        Preliminary_Net_Amount=("Line_Amount", "sum")
    )
    .reindex(date_range)
)
daily_revenue_coverage.index.name = "Date"
recorded_days = daily_revenue_coverage.dropna(subset=["Total_Records"])
coverage_summary = pd.DataFrame({
    "Metric": [
        "Days with Recorded Transactions",
        "Days with Positive-Amount Records",
        "Days with Negative-Amount Records",
        "Days with Zero-Amount Records",
        "Days with Cancellation Records",
        "Days with Negative Preliminary Net Amount",
        "Days with Zero Preliminary Net Amount",
        "Days with Positive Preliminary Net Amount"
    ],
    "Value": [
        len(recorded_days),
        (recorded_days["Positive_Amount_Records"] > 0).sum(),
        (recorded_days["Negative_Amount_Records"] > 0).sum(),
        (recorded_days["Zero_Amount_Records"] > 0).sum(),
        (recorded_days["Cancellation_Records"] > 0).sum(),
        (recorded_days["Preliminary_Net_Amount"] < 0).sum(),
        (recorded_days["Preliminary_Net_Amount"] == 0).sum(),
        (recorded_days["Preliminary_Net_Amount"] > 0).sum()
    ]
})
display(coverage_summary)
print("\nPreliminary Daily Net Amount Distribution:")
display(
    recorded_days["Preliminary_Net_Amount"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame()
)

,Metric,Value
0,Days with Recorded Transactions,305
1,Days with Positive-Amount Records,305
2,Days with Negative-Amount Records,302
3,Days with Zero-Amount Records,234
4,Days with Cancellation Records,302
5,Days with Negative Preliminary Net Amount,1
6,Days with Zero Preliminary Net Amount,0
7,Days with Positive Preliminary Net Amount,304



Preliminary Daily Net Amount Distribution:


,Preliminary_Net_Amount
count,305.00
mean,"31,959.83"
std,"17,414.26"
min,"-1,566.23"
25%,"20,728.14"
50%,"27,978.41"
75%,"42,912.40"
90%,"56,905.16"
95%,"62,918.36"
99%,"78,351.25"


**Key Observations**
- **All 305 recorded transaction days contain positive-amount records**, confirming that every observed trading day includes financially meaningful positive transaction activity.
- **Negative-amount records and cancellation transactions occur on 302 of the 305 recorded days**, demonstrating that cancellations and other deductions are widespread rather than isolated events.
- **Zero-amount records occur on 234 days**, indicating that non-revenue-generating entries are present across much of the observation period.
- **Preliminary daily net amounts are positive on 304 days and negative on one day, with no zero-net days**, highlighting an exceptional negative-revenue date that requires investigation before finalizing the forecasting target.
- **Daily preliminary net amounts average 31,959.83, with a median of 27,978.41 and a maximum of 112,141.11**, demonstrating substantial variation in recorded financial activity across trading days.

#### Investigating the Negative Daily Net Amount

In [38]:
negative_net_dates = recorded_days.loc[
    recorded_days["Preliminary_Net_Amount"] < 0
].index
negative_day_records = revenue_coverage_df.loc[
    revenue_coverage_df["Transaction_Date"].isin(negative_net_dates)
].copy()
negative_day_summary = (
    negative_day_records.groupby("Transaction_Date")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Positive_Amount=("Line_Amount", lambda x: x[x > 0].sum()),
        Negative_Amount=("Line_Amount", lambda x: x[x < 0].sum()),
        Zero_Amount_Records=("Line_Amount", lambda x: (x == 0).sum()),
        Cancellation_Records=("Is_Cancellation", "sum"),
        Preliminary_Net_Amount=("Line_Amount", "sum")
    )
)
display(negative_day_summary)
print("\nLargest Negative Line Amounts:")
display(
    negative_day_records.nsmallest(10, "Line_Amount")[
        [
            "InvoiceNo", "StockCode", "Description",
            "Quantity", "UnitPrice", "Line_Amount"
        ]
    ]
)
print("\nNet Amount by Stock Code:")
display(
    negative_day_records.groupby("StockCode")["Line_Amount"]
    .sum()
    .sort_values()
    .head(10)
    .rename("Net_Line_Amount")
    .reset_index()
)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Zero_Amount_Records,Cancellation_Records,Preliminary_Net_Amount
Transaction_Date,,,,,,,
2011-01-05,1743,76,"32,634.47","-34,200.70",4,73,"-1,566.23"



Largest Negative Line Amounts:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,Line_Amount
43702,C540117,AMAZONFEE,AMAZON FEE,-1,"16,888.02","-16,888.02"
43703,C540118,AMAZONFEE,AMAZON FEE,-1,"16,453.71","-16,453.71"
44152,C540155,72802B,OCEAN SCENT CANDLE IN JEWELLED BOX,-54,3.81,-205.74
44302,C540164,C2,CARRIAGE,-1,50.00,-50.00
43783,C540142,22960,JAM MAKING SET WITH JARS,-12,3.75,-45.00
44495,C540177,22781,GUMBALL MAGAZINE RACK,-4,7.65,-30.60
44308,C540164,22246,MAGIC GARDEN FELT GARLAND,-12,1.95,-23.40
44405,C540171,D,Discount,-1,22.97,-22.97
44212,C540158,22423,REGENCY CAKESTAND 3 TIER,-2,10.95,-21.90
44205,C540158,22968,ROSE COTTAGE KEEPSAKE BOX,-2,9.95,-19.90



Net Amount by Stock Code:


,StockCode,Net_Line_Amount
0,AMAZONFEE,"-33,341.73"
1,72802B,-201.49
2,C2,-50.00
3,22960,-32.25
4,22781,-30.60
5,35400,-26.85
6,22246,-23.40
7,D,-22.97
8,22968,-19.90
9,22942,-17.00


**Key Observations**
- **January 5, 2011 is the only recorded date with a negative preliminary net amount (-1,566.23)**, despite containing 1,743 transaction records across 76 unique invoices.
- **Positive transaction amounts total 32,634.47, while negative amounts total -34,200.70**, demonstrating that deductions exceeded positive financial activity on this date.
- **Two AMAZONFEE cancellation entries contribute -33,341.73**, accounting for approximately 97.49% of the day's total negative amounts and representing the primary cause of the negative daily balance.
- **Excluding these two Amazon fee entries would increase the day's preliminary net amount to 31,775.50**, indicating that the negative balance is primarily driven by non-merchandise accounting adjustments rather than ordinary retail sales activity.
- **The findings reinforce the need to distinguish merchandise transactions from fees, discounts, and other accounting adjustments** when defining the final daily sales revenue target for forecasting.